# Module 2 Exercises


In [ ]:
import os
from typing import Dict, List, Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from openai import AsyncOpenAI
from agents import (
    Agent,
    Runner,
    trace,
    function_tool,
    OpenAIChatCompletionsModel,
    input_guardrail,
    output_guardrail,
    GuardrailFunctionOutput,
)


In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

print("OPENAI_API_KEY set:", bool(openai_api_key))
print("GOOGLE_API_KEY set:", bool(google_api_key))
print("DEEPSEEK_API_KEY set:", bool(deepseek_api_key))
print("GROQ_API_KEY set:", bool(groq_api_key))
print("OPENROUTER_KEY set:", bool(openrouter_api_key))


In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"



def normalize_chat_model(model_name: str, base_url: str | None, api_key: str | None):
    """Return model wrapper when an API key exists."""
    print(model_name, base_url, api_key)
    if not api_key:
        return None
    if base_url:
        client = AsyncOpenAI(base_url=base_url, api_key=api_key)
    else:
        client = AsyncOpenAI(api_key=api_key)
    return OpenAIChatCompletionsModel(model=model_name, openai_client=client)


models = {
    "openai": "openai/gpt-4o-mini",
    "gemini": "google/gemini-2.0-flash-001",
    "deepseek": "deepseek/deepseek-chat",
    "groq": "meta-llama/llama-3.3-70b-instruct",
}

model_registry = {
    "openai": normalize_chat_model(models["openai"], OPENROUTER_BASE_URL, openrouter_api_key),
    "gemini": normalize_chat_model(models["gemini"], OPENROUTER_BASE_URL, openrouter_api_key),
    "deepseek": normalize_chat_model(models["deepseek"], OPENROUTER_BASE_URL, openrouter_api_key),
    "groq": normalize_chat_model(models["groq"], OPENROUTER_BASE_URL, openrouter_api_key),
}

available_models = {k: v for k, v in model_registry.items() if v is not None}
if not available_models:
    raise RuntimeError("No model keys found. Add at least one provider API key in .env.")

available_models


In [ ]:
class ColdEmail(BaseModel):
    """Structured email draft."""

    subject: str = Field(min_length=8, max_length=120)
    preview_text: str = Field(min_length=12, max_length=160)
    body_text: str = Field(min_length=120)
    cta: str = Field(min_length=8, max_length=120)
    tone: Literal["serious", "engaging", "concise", "playful"]


class NameCheckOutput(BaseModel):
    """Personal name detection output."""

    is_name_in_message: bool
    name: str


class SafetyReviewOutput(BaseModel):
    """Output policy review result."""

    blocked: bool
    reason: str


class Contact(BaseModel):
    name: str
    email: str
    company: str


class EmailPayload(BaseModel):
    to: str
    subject: str
    html: str

In [ ]:
SALES_CONTEXT = """
You work at ComplAI, an AI tool that helps teams prepare for SOC2 audits.
Write outbound emails to B2B prospects. Keep claims realistic and specific.
""".strip()


def make_sales_agent(name: str, tone: str, model):
    """Create one model-specific sales drafter."""
    instructions = f"""
{SALES_CONTEXT}
You write in a {tone} style.
Return output in the schema fields for subject, preview_text, body_text, cta, and tone.
""".strip()
    return Agent(name=name, instructions=instructions, model=model, output_type=ColdEmail)


sales_agents = []
for provider, model in available_models.items():
    tone_map = {
        "openai": "serious",
        "gemini": "engaging",
        "deepseek": "concise",
        "groq": "playful",
    }
    tone = tone_map[provider]
    print(provider, tone, model)
    sales_agents.append(make_sales_agent(f"{provider}_sales_agent", tone, model))

sales_tools = [
    agent.as_tool(
        tool_name=agent.name,
        tool_description="Generate a structured cold email draft",
    )
    for agent in sales_agents
]

len(sales_tools), sales_tools, sales_agents


In [ ]:
def _render_email_logic(email: ColdEmail, recipient_name: str, company: str) -> str:
    """The actual rendering logic, safe to call from other functions."""
    return f"""
    <html>
      <body style='font-family: Arial, sans-serif; max-width: 680px;'>
        <p>Hi {recipient_name},</p>
        <p>{email.body_text}</p>
        <p><strong>{email.cta}</strong></p>
        <p>Best,<br/>ComplAI Team</p>
        <hr/>
        <small>For {company} | Preview: {email.preview_text}</small>
      </body>
    </html>
    """.strip()


@function_tool
def get_target_contacts(segment: str = "soc2_readiness") -> List[Dict[str, str]]:
    """Return a small target contact list."""
    data = {
        "soc2_readiness": [
            {"name": "Head of Engineering", "email": "eng-lead@example.com", "company": "Northwind"},
            {"name": "VP Security", "email": "security-vp@example.com", "company": "Contoso"},
            {"name": "Compliance Director", "email": "compliance@example.com", "company": "Fabrikam"},
        ]
    }
    return data.get(segment, [])


@function_tool
def render_html_email(email: ColdEmail, recipient_name: str, company: str) -> str:
    """Render HTML from structured fields."""
    return _render_email_logic(email, recipient_name, company)


@function_tool
def build_mail_merge_plan(email: ColdEmail, contacts: List[Contact]) -> List[Dict[str, str]]:
    """Build recipient-specific payloads."""
    plan = []
    for contact in contacts:
        plan.append(EmailPayload(
            to=contact.email,
            subject=email.subject,
            html=_render_email_logic(email, contact.name, contact.company)
        ))
    return plan


@function_tool
def send_mail_merge_dry_run(payloads: List[EmailPayload]) -> Dict[str, str | int]:
    """Dry run send operation."""
    # My sendgrid is unavailable atm
    return {"status": "dry_run_ok", "emails_prepared": len(payloads), "payloads": payloads}


In [ ]:
name_guardrail_agent = Agent(
    name="name_guardrail",
    instructions=(
        "Check if the input asks to include a personal person name or direct private identifier. "
        "Set is_name_in_message=True when personal naming is requested."
    ),
    output_type=NameCheckOutput,
    model=model_registry["openai"],
)


output_guardrail_agent = Agent(
    name="output_guardrail",
    instructions=(
        "Review proposed outbound email content. Block if it contains deceptive claims, "
        "fabricated guarantees, or requests for sensitive credentials."
    ),
    output_type=SafetyReviewOutput,
    
    model=model_registry["openai"],
)


@input_guardrail
async def guardrail_against_personal_name(ctx, agent, user_input):
    """Block requests that include personal naming."""
    result = await Runner.run(name_guardrail_agent, user_input, context=ctx.context)
    name_check = result.final_output.model_dump()
    blocked = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(
        output_info={"name_check": name_check},
        tripwire_triggered=blocked,
    )


@output_guardrail
async def outbound_safety_guardrail(ctx, agent, output):
    """Block unsafe outbound email content."""
    review = await Runner.run(output_guardrail_agent, str(output), context=ctx.context)
    safety_review = review.final_output.model_dump()
    blocked = review.final_output.blocked
    return GuardrailFunctionOutput(
        output_info={"safety_review": safety_review},
        tripwire_triggered=blocked,
    )


In [ ]:
sales_manager_instructions = """
You are the Sales Manager.

Steps:
1. Use all available sales agent tools to produce candidate drafts.
2. Choose one final draft with the best relevance and clarity.
3. Pull contacts using get_target_contacts.
4. Build recipient payloads with build_mail_merge_plan.
5. Execute send_mail_merge_dry_run.

Rules:
- Do not invent recipient data.
- Do not send directly; use dry run sender only.
- Return a short summary of why the chosen draft won.
""".strip()

manager_tools = [*sales_tools, get_target_contacts, build_mail_merge_plan, send_mail_merge_dry_run]

sales_manager = Agent(
    name="sales_manager",
    instructions=sales_manager_instructions,
    tools=manager_tools,
    model=model_registry["openai"],
    input_guardrails=[guardrail_against_personal_name],
    output_guardrails=[outbound_safety_guardrail],
)


In [ ]:
safe_message = "Create a SOC2 outreach email for startup CTOs and prepare a dry-run mail merge."

with trace("module2_exercise_mail_merge_guardrails"):
    safe_result = await Runner.run(sales_manager, safe_message)

print(safe_result.final_output)


In [ ]:
blocked_message = "Create and send a SOC2 outreach email addressed personally to Alice Johnson."

with trace("module2_exercise_guardrail_block_test"):
    blocked_result = await Runner.run(sales_manager, blocked_message)

print(blocked_result.final_output)
